# 02 · Pocket Graph Construction & Attentive GNN Training

**Pipeline stage:** Pocket graphs → GNN training → druggability ranking

This notebook covers:
1. Converting pocket dicts into PyTorch Geometric `Data` objects
2. Training an **Attentive Pocket GNN** with focal loss
3. Platt calibration and threshold selection
4. Ranking all pockets by predicted druggability probability

> **Prerequisite:** run `01_bioemu_pocket_detection.ipynb` first to generate `outputs/all_pockets.pkl`.


In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader

from bioemu_pocket.graph_builder import PocketGraphBuilder
from bioemu_pocket.model import AttentivePocketGNN, FocalLoss
from bioemu_pocket.trainer import Trainer, build_group_splits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

with open("./outputs/all_pockets.pkl", "rb") as f:
    ALL_POCKETS = pickle.load(f)

# Flatten all proteins into a single list
pocket_data = [p for pockets in ALL_POCKETS.values() for p in pockets]
print(f"Total pockets loaded: {len(pocket_data)}")

## 1 · Graph Construction

Each pocket becomes a **fully-connected graph** (within 10 Å) where:

### Node features (30-dim)

| Dims | Feature |
|------|---------|
| 0–19 | One-hot amino-acid type |
| 20 | Kyte–Doolittle hydrophobicity / 5 |
| 21–25 | Aromatic / polar / charged / small / branched flags |
| 26–28 | Relative depth, neighbour count, distance to centroid |
| 29 | log(1 + neighbour count) |

### Edge features (4-dim)

| Dim | Feature |
|-----|---------|
| 0 | Distance kernel: 1/(1+d) |
| 1 | Hydrophobic potential: h_i × h_j / 25 |
| 2 | Electrostatic: –q_i × q_j |
| 3 | Aromatic–aromatic flag |

### Global features (11-dim)

Normalised pocket-level descriptors: size, convex volume, surface area, compactness, charge density, helix/sheet propensity, flexibility, hydrophobicity, aromatic content, std depth.

### Druggability target

A heuristic score weighted across five criteria (Halgren 2009; Schmidtke & Barril 2010):

$$s = 0.20 \cdot s_{\text{size}} + 0.20 \cdot s_{\text{vol}} + 0.20 \cdot s_{\text{hydro}} + 0.15 \cdot s_{\text{compact}} + 0.10 \cdot s_{\text{charge}} + 0.15 \cdot p_{\text{persist}}$$

where each $s_{\cdot}$ is a Gaussian or clamped score and $p_{\text{persist}} \in \{1.0, 0.3\}$. The score is binarised at 0.5 before training.


In [ ]:
builder = PocketGraphBuilder(pocket_data)
graphs = builder.build_all()

# Binarise target at 0.5
for g in graphs:
    g.y = torch.tensor([1.0 if g.y.item() >= 0.5 else 0.0], dtype=torch.float32)

node_dim = graphs[0].x.shape[1]
edge_dim = graphs[0].edge_attr.shape[1]
global_dim = graphs[0].u.shape[0]
print(f"node_dim={node_dim}  edge_dim={edge_dim}  global_dim={global_dim}")

## 2 · Dataset Splitting

We use **5-fold GroupKFold** with groups assigned by conformer ID // 10, ensuring that
conformers of the same protein sequence do not appear in both train and validation.
This avoids the common leakage problem in conformational ensemble datasets.


In [ ]:
y_values = np.array([g.y.item() for g in graphs])
train_idx, val_idx, test_idx, groups = build_group_splits(graphs, y_values, pocket_data)

train_set = [graphs[i] for i in train_idx]
val_set = [graphs[i] for i in val_idx]
test_set = [graphs[i] for i in test_idx]

print(f"Train: {len(train_set)}  Val: {len(val_set)}  Test: {len(test_set)}")
print(f"Class balance (train): {np.mean([g.y.item() for g in train_set]):.2%} positive")

# Weighted sampler to counteract class imbalance
train_y = np.array([g.y.item() for g in train_set])
w_pos = 0.5 / max(1, train_y.sum())
w_neg = 0.5 / max(1, (1 - train_y).sum())
weights = np.where(train_y == 1.0, w_pos, w_neg).astype(np.float32)
sampler = torch.utils.data.WeightedRandomSampler(
    weights, len(train_set), replacement=True
)

train_loader = DataLoader(train_set, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

## 3 · Model & Training

### Architecture

```
NodeEncoder(Lin→ReLU→Drop)
  → GAT-1 (4 heads, 128d) + GraphNorm
  → GAT-2 (4 heads, 128d) + GraphNorm + residual
  → GAT-3 (1 head,  128d) + GraphNorm
  → [MeanPool | MaxPool | SumPool]
        ↕ concat
  GlobalEncoder(Lin→LN→ReLU→Lin, 32d)
  → FusionMLP(256→128→64→1)
```

### Focal Loss

Standard BCE is modified to down-weight easy examples:

$$\mathcal{L}_{\text{focal}} = -\alpha_t (1 - p_t)^{\gamma} \log p_t$$

with $\alpha = 0.85$ (upweights rare druggable pockets), $\gamma = 2.0$.

### GraphNorm

GraphNorm normalises node features *per graph* rather than per batch,
preventing large-graph statistics from dominating small graphs:

$$\hat{h}_i = \frac{h_i - \alpha \cdot \mu_G}{\sigma_G + \epsilon}$$

where $\mu_G, \sigma_G$ are the per-graph mean and std, and $\alpha$ is a
learnable per-instance shift (Cai et al., NeurIPS 2021).


In [ ]:
model = AttentivePocketGNN(
    node_dim=node_dim,
    edge_dim=edge_dim,
    global_dim=global_dim,
    hidden_dim=128,
    heads=4,
    dropout=0.3,
)
loss_fn = FocalLoss(alpha=0.85, gamma=2.0)
trainer = Trainer(model, device, loss_fn, learning_rate=1e-3, weight_decay=1e-4)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

history = trainer.fit(train_loader, val_loader, epochs=80, early_stopping_patience=12)

## 4 · Evaluation & Calibration

After training we apply **Platt scaling** (logistic regression on raw logits) to
calibrate probabilities.  The optimal classification threshold is chosen by:

1. Sweep candidate thresholds over unique predicted probabilities.
2. Select the threshold maximising F1 subject to precision ≥ 0.50.
3. Fall back to Youden's J (max TPR − FPR) if no threshold satisfies the constraint.


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], alpha=0.9)
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Training Loss")
axes[0].grid(alpha=0.3)
axes[1].plot(history["val_auroc"], label="AUROC")
axes[1].plot(history["val_auprc"], label="AUPRC")
axes[1].set(xlabel="Epoch", ylabel="Metric", title="Validation Metrics")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calibration
platt, best_thr, (vp_cal, vt) = trainer.calibrate(val_loader)
print(f"Calibrated threshold: {best_thr:.3f}")

# Test evaluation
test_logits, test_tgts = trainer.evaluate_logits(test_loader)
test_probs = platt.predict_proba(test_logits.reshape(-1, 1))[:, 1]

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
)

tauroc = (
    roc_auc_score(test_tgts, test_probs) if len(set(test_tgts)) > 1 else float("nan")
)
tauprc = average_precision_score(test_tgts, test_probs)
tpreds = (test_probs >= best_thr).astype(int)
print(
    f"Test  AUROC={tauroc:.3f}  AUPRC={tauprc:.3f}  F1={f1_score(test_tgts, tpreds):.3f}"
)

## 5 · Pocket Ranking

In [ ]:
model.eval()
all_scores = []
with torch.no_grad():
    for g in graphs:
        batch = Batch.from_data_list([g]).to(device)
        logit = model(batch).item()
        prob = platt.predict_proba(np.array([[logit]]))[0, 1]
        all_scores.append(float(prob))

for p, s in zip(pocket_data[: len(all_scores)], all_scores):
    p["druggability_score"] = s

ranked = sorted(
    pocket_data[: len(all_scores)], key=lambda x: x["druggability_score"], reverse=True
)

print("\nTOP 10 PREDICTED DRUGGABLE POCKETS")
print(f"{'Rank':>4}  {'Score':>6}  {'Size':>4}  {'Conf':>4}  {'Status':<10}  Hydro")
for i, p in enumerate(ranked[:10], 1):
    status = "Cryptic" if p.get("is_cryptic") else "Persistent"
    print(
        f"{i:>4}  {p['druggability_score']:.3f}  {p['size']:>4}  {p['conf_id']:>4}  {status:<10}  {p['hydrophobicity']:.2f}"
    )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(all_scores, bins=30, alpha=0.7, edgecolor="black")
axes[0].axvline(
    np.median(all_scores),
    color="red",
    ls="--",
    label=f"median={np.median(all_scores):.3f}",
)
axes[0].set(xlabel="Druggability Probability", title="Score Distribution")
axes[0].legend()

sizes = [p["size"] for p in ranked]
scores = [p["druggability_score"] for p in ranked]
axes[1].scatter(sizes, scores, alpha=0.5, s=15)
axes[1].set(xlabel="Pocket Size (residues)", ylabel="Score", title="Score vs. Size")
axes[1].grid(alpha=0.3)

persist = [p["druggability_score"] for p in ranked if not p.get("is_cryptic")]
cryptic = [p["druggability_score"] for p in ranked if p.get("is_cryptic")]
axes[2].boxplot([persist, cryptic], labels=["Persistent", "Cryptic"])
axes[2].set(ylabel="Score", title="Score by Pocket Type")
axes[2].grid(alpha=0.3)

plt.suptitle("Pocket Druggability Predictions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6 · Save Results

In [ ]:
from pathlib import Path
import pickle

out = Path("./outputs")
out.mkdir(exist_ok=True)

top20 = pd.DataFrame(
    [
        {
            "rank": i,
            "conf_id": p["conf_id"],
            "size": p["size"],
            "score": p["druggability_score"],
            "hydrophobicity": p["hydrophobicity"],
            "is_cryptic": p.get("is_cryptic", False),
            "volume": p["volume"],
            "depth": p["depth"],
        }
        for i, p in enumerate(ranked[:20], 1)
    ]
)
top20.to_csv(out / "top20_pockets.csv", index=False)
torch.save(
    {"model": model.state_dict(), "platt": platt, "threshold": best_thr},
    out / "trained_model.pt",
)
print("Saved top20_pockets.csv and trained_model.pt")